### Регрессия: все операции resist_id, тест на 2025 году

Вариант на основе regression_forecast.ipynb:
* На вход — **все** operation_id выбранного resist_id
* Тест — **20% последних месяцев** 2025 года; обучение — на всех предыдущих годах
* LinearRegression + метрики Точность, Стабильность, Качество

In [ ]:
# импорт библиотек
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
%matplotlib inline
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")

#### Конфигурация

Выбираем **один** resist_id; используются **все** его operation_id. Тест — 20% последних месяцев 2025.

In [ ]:
# --- Выбор resist_id (используются все operation_id) ---
RESIST_ID = 124
TEST_YEAR = 2025
TEST_SHARE_2025 = 0.2  # тест — 20% последних месяцев 2025

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]

#### Загрузка данных, оценка и подготовка

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_division_title(div_json: str):
    if pd.isna(div_json):
        return None
    try:
        obj = json.loads(div_json)
    except Exception:
        return None
    return obj.get("name")


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def get_calc_value(calc_obj: dict, key: str):
    val = calc_obj.get(key)
    if isinstance(val, dict):
        return val.get("value")
    return val


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    """Разворачивает calculate_parameter в плоский словарь: все поля → колонки calc_<key>."""
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


# Загрузка CSV
dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(
        f"Файлы данных не найдены в {RAW_DIR}. "
        "Убедитесь, что папка data/raw существует и содержит data-2022.csv и др."
    )

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "division_title": parse_division_title(row.get("division_name")),
            "id": row.get("id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

# Приведение calc_* колонок к числу (где возможно)
calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print("Развёрнутые поля calculate_parameter:", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

# Фильтр: только выбранный resist_id (все operation_id)
df = df_model39[df_model39["resist_id"] == RESIST_ID].copy()

if df.empty:
    raise ValueError(
        f"Нет данных для resist_id={RESIST_ID}. "
        f"Доступные resist_id: {sorted(df_model39['resist_id'].dropna().unique()[:10].tolist())}..."
    )

# One-hot для operation_id (модель должна различать операции); сохраняем operation_id для Excel
df["operation_id"] = df["operation_id"].astype(str)
op_id_for_grouping = df["operation_id"].copy()
df = pd.get_dummies(df, columns=["operation_id"], prefix="op")
df["operation_id"] = op_id_for_grouping

# Feature Engineering (как в catboost_per_resist_id)
# group = df.groupby("operation_id")
# df["lag_1"] = group["fact_value"].shift(1)
# df["lag_12"] = group["fact_value"].shift(12)
# df["roll_mean_3"] = group["fact_value"].transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
# df["roll_mean_6"] = group["fact_value"].transform(lambda s: s.shift(1).rolling(6, min_periods=6).mean())
# df["trend"] = (df["lag_1"] / df["roll_mean_3"]).replace([np.inf, -np.inf], 0)

num_cols_clean = df.select_dtypes(include=["number"]).columns
df[num_cols_clean] = df[num_cols_clean].fillna(0)
df = df.infer_objects(copy=False)

df.head()

#### Проверка пропусков данных

In [ ]:
# проверка пропусков значений
if df.isnull().sum().sum() != 0:
    print("Обнаружены пропуски:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    df = df.fillna(0)
else:
    print("dataset не содержит пустых значений")

#### Оценка датасета и построение графиков

In [ ]:
df.describe().T

In [ ]:
# График: fact_value (сумма по операциям) по времени
df_plot = df.copy()
df_plot["date_plot"] = pd.to_datetime(df_plot["date"])
agg = df_plot.groupby("date_plot")["fact_value"].sum()
plt.figure(figsize=(10, 4))
plt.plot(agg.index, agg.values, marker="o", markersize=4)
plt.xlabel("Дата")
plt.ylabel("fact_value (сумма по операциям)")
plt.title(f"Динамика fact_value для resist_id={RESIST_ID} (все операции)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Распределение целевой переменной
sns.histplot(df["fact_value"], bins=15, kde=True)
plt.xlabel("fact_value")
plt.ylabel("Частота")
plt.title("Распределение количества операций")
plt.show()

#### Разделение: обучение до 2025, тест — 20% последних месяцев 2025

In [ ]:
# Признаки: базовые + one-hot operation_id
BASE_FEATURE_COLS = ["month", "year", "workDayMonth_2", "calculate_minute"]
OP_COLS = [c for c in df.columns if c.startswith("op_")]
FEATURE_COLS = BASE_FEATURE_COLS + OP_COLS
TARGET_COL = "fact_value"

train_df = df[df["year"] < TEST_YEAR].copy()

# 20% последних месяцев 2025
df_2025 = df[df["year"] == TEST_YEAR].copy()
months_2025 = sorted(df_2025["month"].unique())
n_take = max(1, int(len(months_2025) * TEST_SHARE_2025))
test_months = months_2025[-n_take:]
test_df = df_2025[df_2025["month"].isin(test_months)].copy()

if train_df.empty:
    raise ValueError(f"Нет данных для обучения (год < {TEST_YEAR})")
if test_df.empty:
    raise ValueError(f"Нет данных за {TEST_YEAR} для теста. Доступные годы: {sorted(df['year'].unique())}")

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]

print(f"Обучающая выборка: {len(X_train)} строк (до {TEST_YEAR})")
print(f"Тестовая выборка: {len(X_test)} строк ({TEST_SHARE_2025*100:.0f}% последних месяцев {TEST_YEAR}: {test_months})")

#### Обучение модели LinearRegression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

#### Оценка точности предсказания

In [ ]:
y_pred = model.predict(X_test)
y_pred = np.maximum(y_pred, 0)  # предсказания не должны быть отрицательными

df_test = test_df.copy()
df_test["predict_linear"] = y_pred

y_true = np.asarray(y_test, dtype=float)
y_pred = np.asarray(y_pred, dtype=float)

# Точность по строке: 100 - MAPE%, с ограничением [0, 100]
acc = np.empty_like(y_true)
mask_zero = y_true == 0
acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
mask_ok = ~mask_zero
if np.any(mask_ok):
    acc[mask_ok] = np.clip(
        100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
        0, 100,
    )

# Точность — среднее по accuracy_pct
tochnost = float(np.mean(acc))
# Стабильность — (1 - CV) * 100%, CV = σ/μ по accuracy_pct
mean_acc = float(np.mean(acc))
stab = 0.0
if mean_acc > 0:
    cv = np.std(acc, ddof=0) / mean_acc
    stab = max(0.0, (1 - cv)) * 100
# Качество — гармоническое среднее Точности и Стабильности
kachestvo = 0.0
if tochnost + stab > 0:
    kachestvo = 2 * tochnost * stab / (tochnost + stab)

print(f"Точность, %:   {tochnost:.2f}")
print(f"Стабильность, %: {stab:.2f}")
print(f"Качество, %:   {kachestvo:.2f}")

# График: факт vs предсказание
plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="Идеальная линия")
plt.xlabel("Фактическое значение")
plt.ylabel("Предсказание")
plt.title("Факт vs предсказание LinearRegression")
plt.legend()
plt.tight_layout()
plt.show()

#### Выводы по работе

* **Точность** — среднее по «Точность прогноза» (100% − MAPE) по строкам. Чем выше, тем меньше относительная ошибка.
* **Стабильность** — (1 − CV) × 100%, где CV = σ/μ по «Точность прогноза». Чем меньше разброс точности, тем стабильнее.
* **Качество** — гармоническое среднее Точности и Стабильности. Падение любого показателя снижает рейтинг.

Сравнение можно проводить, меняя `RESIST_ID` или добавляя другие признаки.

In [ ]:
def calc_metrics_for_forecast(df_sub: pd.DataFrame, forecast_col: str) -> tuple[float, float, float]:
    """Точность, Стабильность, Качество по df с fact_value и forecast_col."""
    if df_sub.empty:
        return 0.0, 0.0, 0.0
    y_true = df_sub["fact_value"].astype(float).values
    y_pred = df_sub[forecast_col].astype(float).fillna(0).values
    y_pred = np.maximum(y_pred, 0)
    acc = np.empty_like(y_true, dtype=float)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = float(np.mean(acc))
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = max(0.0, (1 - cv)) * 100
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = 2 * tochnost * stab / (tochnost + stab)
    return round(tochnost, 2), round(stab, 2), round(kachestvo, 2)


# Таблица по операциям
rows_op = []
for (rid, op_id), grp in df_test.groupby(["resist_id", "operation_id"]):
    t1, s1, q1 = calc_metrics_for_forecast(grp, "fact_predict")
    t2, s2, q2 = calc_metrics_for_forecast(grp, "predict_linear")
    div = grp["division_title"].iloc[0] if "division_title" in grp.columns else ""
    rows_op.append({
        "resist_id": rid,
        "operation_id": op_id,
        "division_title": div,
        "Точность (fact_predict)": t1,
        "Стабильность (fact_predict)": s1,
        "Качество (fact_predict)": q1,
        "Точность (прогноз)": t2,
        "Стабильность (прогноз)": s2,
        "Качество (прогноз)": q2,
    })

# Таблица по resist_id в целом
rows_rid = []
for rid, grp in df_test.groupby("resist_id"):
    t1, s1, q1 = calc_metrics_for_forecast(grp, "fact_predict")
    t2, s2, q2 = calc_metrics_for_forecast(grp, "predict_linear")
    div = grp["division_title"].iloc[0] if "division_title" in grp.columns else ""
    rows_rid.append({
        "resist_id": rid,
        "division_title": div,
        "Точность (fact_predict)": t1,
        "Стабильность (fact_predict)": s1,
        "Качество (fact_predict)": q1,
        "Точность (прогноз)": t2,
        "Стабильность (прогноз)": s2,
        "Качество (прогноз)": q2,
    })

df_metrics_op = pd.DataFrame(rows_op)
df_metrics_rid = pd.DataFrame(rows_rid)

# Детали тестовой выборки: даты и значения
df_detail = df_test[["resist_id", "operation_id", "division_title", "year", "month", "fact_value", "fact_predict", "predict_linear"]].copy()
df_detail = df_detail.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
df_detail = df_detail.rename(columns={
    "fact_value": "fact_value (факт)",
    "fact_predict": "fact_predict (кол-во)",
    "predict_linear": "прогноз (кол-во)",
})

out_path = Path("../data/processed/regression_forecast_all_ops_metrics.xlsx")
out_path.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    df_detail.to_excel(w, sheet_name="Данные теста", index=False)
    df_metrics_op.to_excel(w, sheet_name="По операциям", index=False)
    df_metrics_rid.to_excel(w, sheet_name="По resist_id", index=False)

print(f"Excel сохранён: {out_path.resolve()}")